# 02_scoring: Disease Gene Scoring and Statistics

This notebook demonstrates the scoring pipeline for a single disease.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from load_data import load_allen_expression, load_gwas_catalog, match_genes_to_atlas, zscore_expression
from gene_lists import build_gene_lists
from scoring import compute_regional_scores, compute_donor_scores, compute_consistency_metric
from stats import run_permutation_test, compute_effect_sizes
from validation import validate_disease, load_expected_regions

In [ ]:
# Load data
expression_matrix, donor_matrices = load_allen_expression(atlas="desikan-killiany")

# Build gene lists for Parkinson's
strict_genes, broad_genes = build_gene_lists(
    "Parkinson",
    expression_matrix,
    pvalue_threshold_strict=5e-8,
    pvalue_threshold_broad=1e-5,
    force_refresh=False
)

In [ ]:
# Compute scores for strict list
strict_scores = compute_regional_scores(expression_matrix, strict_genes)
strict_donor_scores = compute_donor_scores(donor_matrices, strict_genes)
strict_consistency = compute_consistency_metric(strict_donor_scores)

print(f"Strict genes: {len(strict_genes)}")
print(f"Top 10 regions by score:")
print(strict_scores.nlargest(10))

In [ ]:
# Run permutation test (using smaller n_sets for demo)
observed, pvals, fdr_results, effects = run_permutation_test(
    expression_matrix,
    strict_genes,
    n_sets=1000,  # Reduced for demo
    random_state=42
)

# Combine results
results = pd.DataFrame({
    "score": observed,
    "pvalue": pvals,
    "fdr": fdr_results["fdr"],
    "significant": fdr_results["significant"],
    "cohens_d": effects["cohens_d"],
})

print(f"Significant regions (FDR < 0.05): {results['significant'].sum()}")
print(results.nlargest(15, "score")[['score', 'pvalue', 'fdr', 'cohens_d', 'significant']])

In [ ]:
# Donor consistency
print("Donor consistency (n positive donors):")
print(strict_consistency.nlargest(10))

# Visualize donor scores for top regions
top_regions = strict_scores.nlargest(5).index
fig, axes = plt.subplots(1, len(top_regions), figsize=(15, 4))
for i, region in enumerate(top_regions):
    axes[i].bar(strict_donor_scores.columns, strict_donor_scores.loc[region])
    axes[i].set_title(region)
    axes[i].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Validation
expected = load_expected_regions()
parkinson_expected = expected[expected["disease"] == "Parkinson"]["region"].tolist()
print(f"Expected regions for Parkinson's: {parkinson_expected}")

validation = validate_disease(
    disease="Parkinson",
    stats_results=results,
    donor_consistency=strict_consistency,
    expression_matrix=expression_matrix,
    strict_genes=strict_genes,
    broad_genes=broad_genes,
    list_type="strict"
)

print(validation[['badge', 'fdr_pass', 'consistency_pass', 'in_expected', 'n_positive_donors']].head(20))

## Next Steps

1. Run with full n_sets=5000 for final results
2. Repeat for Alzheimer's and Huntington's
3. Run negative control (Height)
4. Compare strict vs broad gene lists
5. Generate reports